In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import importlib
import ml_pipeline

In [38]:
train_data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

test_data_pl =  pl.read_csv(r"../data/test.csv",encoding="shift_jis")

### データ確認（ここから）

In [39]:
train_data_pl.select("species number").unique

<bound method DataFrame.unique of shape: (1_322, 1)
┌────────────────┐
│ species number │
│ ---            │
│ i64            │
╞════════════════╡
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ …              │
│ 19             │
│ 19             │
│ 19             │
│ 19             │
│ 19             │
└────────────────┘>

### データ確認（ここまで）

### 各木種類ごとにモデルを作成する：sp=1で確認(ここから)

trainに存在する樹種がtestに存在しないので注意

In [40]:
from ml_pipeline import MoisturePipeline

### 各木種類ごとにモデルを作成するsp=1で確認(ここまで)

#### MLflowを使い各樹ごとモデルのスタッキングを行う

In [41]:
train_df = train_data_pl 
test_df  = test_data_pl

In [43]:
import mlflow
import polars as pl

from ml_pipeline import MoisturePipeline
from stacking import (
    create_oof,
    train_meta_model,
    StackingModel
)

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("moisture_stacking")

models = {}
species_list = train_df["species number"].unique().to_list()

# =========================
# 親run
# =========================
with mlflow.start_run(run_name="parent_run"):

    # =========================
    # ① speciesごとモデル
    # =========================
    for sp in species_list:

        sp_train = train_df.filter(pl.col("species number") == sp)

        if sp_train.height == 0:
            continue

        pipe = MoisturePipeline(species=f"sp_{sp}")

        #樹種ごとにrunを作成する
        with mlflow.start_run(run_name=f"species_{sp}", nested=True):

            rmse = pipe.fit(sp_train)

            mlflow.log_param("species", sp)
            mlflow.log_metric("rmse", rmse)

            # ★ pipeline丸ごと保存
            mlflow.pyfunc.log_model(
                name=f"model_sp_{sp}",
                python_model=pipe   # ←シンプル版
            )

        models[sp] = pipe

    # =========================
    # ② OOF（stacking用）
    # =========================
    X_meta, y = create_oof(train_df, species_list)

    # =========================
    # ③ metaモデル
    # =========================
    meta_model = train_meta_model(X_meta, y)

    # =========================
    # ④ stacking保存
    # =========================
    with mlflow.start_run(run_name="stacking", nested=True):

        mlflow.pyfunc.log_model(
            name="stacking_model",
            python_model=StackingModel(models, meta_model, species_list)
        )

/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/04/02 21:11:23 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


🏃 View run species_1 at: http://mlflow:5000/#/experiments/3/runs/c1f9747b971a4151b5d3bab3db80c664
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run parent_run at: http://mlflow:5000/#/experiments/3/runs/bae79f92b8c243b7af9c7297e0219981
🧪 View experiment at: http://mlflow:5000/#/experiments/3


MlflowException: `python_model` must be a PythonModel instance, callable object, or path to a script that uses set_model() to set a PythonModel instance or callable object.